## Optuna

Hyperparameter tuning framework 
We can do hpt on very big datasets 

## Why?
We use gridsearch CV and randomsearch CV normally so why use optuna 
We decide on a search space where we decide on the params to check
for ex 
in random forest we can use n_estimators(50,250,50) and max_depth(1,5,1)
In grid search CV - it does training on all possible params (Dumb af)
Problem - it is very costly , imagine multiple datasets your pc gonna die lol

To solve this problem  we use RandomSearch(Dumb af)
It does on random combinations ie a set amount of combinations 
Saves computations but kinda tukka

To solve this we use Optuna
it uses a different kind of Search which is Bayesian Search (inspired from probability)
    - max_depth and n_estimator -> accurarcy , then we can say that there is a mathematical rln between accurarcy and these features
    - accuracy = f(max_depth,n_estimator)
    - finds the nature of the graph which would give us maxima 
    - this would approximate the graph with different points so on and so forth

### Some key terms
1) Study - The optimisation session is known as study.
2) Trial - A specific set of hp is selected and run that is it
3) Trial Params - The params in the trial are known as Tp's
4) Objective Function - The function we need to find between the acuuracy and the parameters
5) Sampler -  The algo which dictates how you need to find the next point based on previous points.
    TPE - Tree structured parzen estimator (read research paper)

In [1]:
import optuna
from sklearn.datasets import load_diabetes 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np 

Preprocessing


In [2]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [3]:
df = pd.read_csv(url,names = columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
cols_with_missing_vals = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
df[cols_with_missing_vals]=df[cols_with_missing_vals].replace(0,np.nan)

In [5]:
df.fillna(df.mean(),inplace=True)

In [6]:
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


Data prep

In [7]:
X = df.drop('Outcome' ,axis=1)
y = df['Outcome']

In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [9]:
scaler = StandardScaler()

In [10]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [11]:
from sklearn.ensemble  import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC

from sklearn.model_selection import cross_val_score

In [12]:
# def the obj function
def Objective(trial):
    # suggest values for hyperparams
    n_estimators = trial.suggest_int('n_estimators',50,200)
    max_depth = trial.suggest_int('max_depth',3,20)

    # create random forest classifier with the hyperparams
    model = RandomForestClassifier(
        n_estimators = n_estimators,
        max_depth=max_depth,
        random_state=42
        )
    # perform 3-fold cross-val and calc accuracy
    score = cross_val_score(model,X_train,y_train,cv = 3 ,scoring = 'accuracy').mean()
    return score

FLOW 
Create objective function (would have the params , model , score / loss / weights etc etc)
put it into study
then optimise the study

In [13]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(Objective,n_trials=75)

[I 2026-03-30 22:26:29,690] A new study created in memory with name: no-name-7d4b4004-93e6-4471-80cf-b78794bc6453
[I 2026-03-30 22:26:30,528] Trial 0 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 196, 'max_depth': 14}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-03-30 22:26:30,877] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 81, 'max_depth': 19}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-03-30 22:26:31,945] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 179, 'max_depth': 11}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-03-30 22:26:32,683] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 98, 'max_depth': 18}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-03-30 22:26:33,128] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 75, 'max_depth': 4}. Best is trial 0 with value: 0.77281191

In [14]:
print(study.best_trial.value)
print(study.best_params)

0.7858472998137803
{'n_estimators': 121, 'max_depth': 15}


In [15]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.RandomSampler())
study.optimize(Objective,n_trials=75)

[I 2026-03-30 22:27:05,682] A new study created in memory with name: no-name-ec7f8bc0-1efc-4c0d-80ef-5a9301bb06fd
[I 2026-03-30 22:27:05,944] Trial 0 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 79, 'max_depth': 3}. Best is trial 0 with value: 0.7560521415270017.
[I 2026-03-30 22:27:06,225] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 80, 'max_depth': 8}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-03-30 22:27:06,792] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 164, 'max_depth': 10}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-03-30 22:27:07,451] Trial 3 finished with value: 0.7783985102420856 and parameters: {'n_estimators': 192, 'max_depth': 16}. Best is trial 3 with value: 0.7783985102420856.
[I 2026-03-30 22:27:08,034] Trial 4 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 168, 'max_depth': 16}. Best is trial 3 with value: 0.77839851

In [16]:
space_search = {
    'n_estimators' : [50,100,150,200],
    'max_depth' : [5,10,15,20]
}
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.GridSampler(space_search))
study.optimize(Objective,n_trials=75)

[I 2026-03-30 22:27:37,821] A new study created in memory with name: no-name-cf285ffc-1837-41e8-a061-f4e6bc8e8891
[I 2026-03-30 22:27:38,229] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-30 22:27:38,777] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-30 22:27:38,983] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-30 22:27:39,307] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-30 22:27:39,666] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

Some Features
1) We have multiple samplers 
    Random Search
    Grid Search
    ....
2) Visualisation we can make OHP (optimisation history plot)
    - Parallel Coodinate plot - rln bw accuracy max deth and estimaors
    - Slice Plot 
    - Contour Plot
    - Importance Plot
3) Define By Run - We can create dynamic search spaces
    - consider i need to train multiple models then we can do it by only doing hpt once
    - we can choose the algos as a hyperparams
    for ex algo = [SVM , XGB , RF , LR]
    then we can have different search spaces for these algos and to choose which search space depends on which algo hyperparameter
    - we can also use distributed computing (kinda use diff comps to train faster)
    

In [17]:
# def the obj function
def Objective_multi(trial):
    classifier_name = trial.suggest_categorical('classifier',['SVM','RandomForest','GradientBoosting'])

    if classifier_name == 'SVM':
    #SVM hyperparams
        c = trial.suggest_float('C',0.1,100,log = True)
        kernel = trial.suggest_categorical('kernel',['linear','rbf','poly','sigmoid'])
        gamma = trial.suggest_categorical('gamma',['scale','auto'])
        
        model = SVC(C=c,kernel = kernel ,gamma = gamma , random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        ) 

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )
        
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

    

In [18]:
study = optuna.create_study(direction='maximize')
study.optimize(Objective_multi, n_trials=100)

[I 2026-03-30 22:27:46,468] A new study created in memory with name: no-name-d2544c41-d2e7-42fc-9fb7-d3bab87a58b9
[I 2026-03-30 22:27:46,894] Trial 0 finished with value: 0.7560521415270017 and parameters: {'classifier': 'RandomForest', 'n_estimators': 132, 'max_depth': 5, 'min_samples_split': 7, 'min_samples_leaf': 9, 'bootstrap': True}. Best is trial 0 with value: 0.7560521415270017.
[I 2026-03-30 22:27:46,946] Trial 1 finished with value: 0.7318435754189944 and parameters: {'classifier': 'SVM', 'C': 37.41211769973061, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 0 with value: 0.7560521415270017.
[I 2026-03-30 22:27:47,346] Trial 2 finished with value: 0.7783985102420857 and parameters: {'classifier': 'RandomForest', 'n_estimators': 146, 'max_depth': 17, 'min_samples_split': 4, 'min_samples_leaf': 9, 'bootstrap': False}. Best is trial 2 with value: 0.7783985102420857.
[I 2026-03-30 22:27:47,377] Trial 3 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM'

In [21]:
print(study.best_params)
print(study.best_trial.value)

{'classifier': 'SVM', 'C': 0.12247555131313376, 'kernel': 'linear', 'gamma': 'scale'}
0.7895716945996275


In [23]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.756052,2026-03-30 22:27:46.469017,2026-03-30 22:27:46.894004,0 days 00:00:00.424987,NaN,True,RandomForest,NaN,NaN,NaN,5.0,9.0,7.0,132.0,COMPLETE
1,1,0.731844,2026-03-30 22:27:46.894004,2026-03-30 22:27:46.946094,0 days 00:00:00.052090,37.412118,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.778399,2026-03-30 22:27:46.947166,2026-03-30 22:27:47.346124,0 days 00:00:00.398958,NaN,False,RandomForest,NaN,NaN,NaN,17.0,9.0,4.0,146.0,COMPLETE
3,3,0.785847,2026-03-30 22:27:47.346124,2026-03-30 22:27:47.377848,0 days 00:00:00.031724,8.822098,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.750466,2026-03-30 22:27:47.377848,2026-03-30 22:27:47.781162,0 days 00:00:00.403314,NaN,NaN,GradientBoosting,NaN,NaN,0.063417,11.0,9.0,5.0,60.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.785847,2026-03-30 22:28:07.325434,2026-03-30 22:28:07.353121,0 days 00:00:00.027687,1.717711,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2026-03-30 22:28:07.354120,2026-03-30 22:28:07.368675,0 days 00:00:00.014555,0.225258,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.767225,2026-03-30 22:28:07.368675,2026-03-30 22:28:08.041392,0 days 00:00:00.672717,NaN,True,RandomForest,NaN,NaN,NaN,5.0,2.0,6.0,180.0,COMPLETE
98,98,0.787709,2026-03-30 22:28:08.043392,2026-03-30 22:28:08.119799,0 days 00:00:00.076407,0.173213,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
